# 다음 Q3 실험: 누락 판별기의 가림 반응 → 원본 후보 부위 확대 → 위치 검토

파일: `lvlm_q3_missing_occlusion_dev_colab.ipynb`

**L4/A100 → 모두 실행 → Drive 연결 허용. 실제 추론 실행용입니다. 새 방법의 실제 효과는 아직 미검증입니다.**
기존 원본 ZIP 두 개를 자동 검색합니다. 앞선 실험을 다시 돌리거나 토큰·추가 라벨을 입력할 필요가 없습니다.

- 이전 실제 dev240의 Q3와 동결 판별기 결과를 내장했습니다. Q1/Q2/gold 추론·재학습·임계값 변경0.
- 기존 Q3가 빈 위치이고 유리 절연체 누락 판별기가 양성인 동일23장에만 새 위치 도구를 호출합니다.
- 원본 전체 사진의9칸을 각각 흐리기/평균색 가리기로 검사: 동결 DINO 특징 최대414회(23×9×2).
- 두 가림 방식에서 점수가 함께 떨어진 최대2칸을 후보로 골라, 원본의 해당 부위를 확대해 LVLM이 위치를 다시 판단합니다.
- 같은 개수·크기의 고정5,2번 칸을 확대하는 대조군도 실행합니다. 새 LVLM은 최대46회(방법23+대조23), 각3이미지/동일예산입니다.
- 두 가림 방식이 동의하는 칸이 없으면 양쪽 모두 추가 판단을 보류합니다. 기존 양성/무효 응답을 사후 재파싱하지 않습니다.
- **가림 반응은 약한 위치 후보이며 정답 위치가 아닙니다.** 배경·가림 인공물 때문에 틀릴 수 있습니다. LVLM은 후보 밖 위치나 빈 집합도 답할 수 있습니다.
- 이 dev에는 결함 위치 정답이 없어 위치 정확도는null입니다. 미탐·정상 오탐/무효율과 원본 후보 crops/maps를 출력합니다.
- 코드/입력/모델/폴더가 같으면 중간 저장부터 재개합니다. 오류만 재시도하며 무효는 오답으로 유지합니다.

In [ ]:
RUN_FOLDER = "lvlm_insplad_results/q3_missing_occlusion_dev240_v1"
SUP_ZIP_PATH = ""
UNSUP_ZIP_PATH = ""

In [ ]:
import base64,hashlib,io,json,sys,zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
SOURCE_SHA256='16efcbbc52b6921837217fdef5568c1d8ea3e13a7d1cc7031add85d221ab04b9'
SOURCE_MANIFEST={'inputs/dataset.json': '301011c1b9e7a4c7f36e1cd1385587b6d0dfa09ff4d4cdc514266cc807113a99', 'inputs/frozen_detector.json': 'd9ff0610d0b74752a2dfc5b53f1cdc1db5e916e17b4260172f1c7abab8c6e7f1', 'inputs/original_features/dev-676efe18c902578c.npy': 'cb8103b0c630d00808dbd3d66c825fdf37a7b63a20f2e92d280926aff492eb47', 'inputs/original_features/dev-2a9f66c0c1d49988.npy': 'af04e02d2cc57392189c157c47a6bbdb50abc43d966ec69b4839e52cff50db42', 'inputs/original_features/dev-65b1682ed9bac3b3.npy': '061371067bcd44ee78037cc74382db23a7a15ba49e6076ec4fe7dfd6a89b3af3', 'inputs/original_features/dev-ab35d50c75c215ff.npy': '1d6c094822ec8fa29e48d1786e33db61cd05faa49ad148c11cb5465b2b2c3cf1', 'inputs/original_features/dev-303182887e9677f5.npy': 'cc4bc97d4f4d02df49791e6a3d8f8694fe65fcf6a2af2e967fc1338210341b2c', 'inputs/original_features/dev-54fb1c557a11f2c0.npy': '8145954e807bc8383c7cbc4164a4d79e97b6217f1d8d420ab751045cede4638d', 'inputs/original_features/dev-d1ef43a89c8b86b0.npy': '21a766daf2ec4930c912581fe836c6c494183e525c29601e67e5d950e6272d46', 'inputs/original_features/dev-5d8b65f59480d260.npy': 'e173a8c0b85851f3484cedf0436d7026cd766bbed3a9deb6adc0491fb57e9c71', 'inputs/original_features/dev-3f1ef393e4238ae8.npy': '43d6de70ee11a9228129573054f6611e76bf668d19ed86861f7dff26134f90c3', 'inputs/original_features/dev-03c0ba2656835309.npy': '5cdef91629b55d0f5e7c832802cec1a92dc40e32bd5c8c92be14608e46a657dd', 'inputs/original_features/dev-349b7d61780001c6.npy': 'fab88bd2b8f3be42d212f60c0e363bf5287ba47bc2b3a6c71a61c0d97b1a6c5a', 'inputs/original_features/dev-adb0de1a52fbdcb7.npy': '5008ed7f9cc7744389093ae4896e042d06dda25c5c5addd212c4ea1058fc54df', 'inputs/original_features/dev-85a1045468839ce4.npy': 'abe754626c8ecf1206dcb14192e243c1c3e165910736be47081f6b368a4c7232', 'inputs/original_features/dev-22b99456fec20a68.npy': '35ae48e6fee00cac11cc40dd7ab956ffc0de22e73407cd03fde3002d5140e2b5', 'inputs/original_features/dev-9ca99e2ac8fb1baf.npy': '143d4216117c70f06753ddddbee27c919f473b187b78e573f51e3b513a95d639', 'inputs/original_features/dev-586fc2ebb11fa743.npy': '890f434b1ecadd680677d03ef61482c6ab4df8852fe1a02feaca2fb192198f12', 'inputs/original_features/dev-b271f28f23168c20.npy': 'fbaa634ff28dea2e36185592579bedc8d820381af20fd21e702e325ee4ebb836', 'inputs/original_features/dev-15b93b38df14da3c.npy': '4211b19c341659423e830f76f2453024296be7042dce439935820c4ed8281cd2', 'inputs/original_features/dev-fcc3deb4a24523e9.npy': '55bb672eafa817df4723e514d56ac4686bec7fcfcd92e50a8b93cfb07d289250', 'inputs/original_features/dev-173a805ee5cfa4b0.npy': '0f3255f6574a8ac7dd26c822a85620cfa424bdd669637abea1cd446a79f267dc', 'inputs/original_features/dev-6382f905cac70bd1.npy': '8663a76dd624a6aee52cc430e9beae4169a7b4e631b8d674b88bba79aac00fbe', 'inputs/original_features/dev-33c9cf299ed1b30e.npy': '6047a22ecd4481ac192587dd6c265089f8db1c049d818d207c6c77d9f48b1b27', 'inputs/original_features/dev-26f757507b3c93c7.npy': '73260de5d115e90ebe8cf66dfe134f273d7d51122cf1617faf228940f4811556', 'scripts/build_explainable_qa_gold.py': '9acdd42e87c15f9da2ba9bb7c3f89d9e77c42d774a8a8d38f6e783ea544d4f0a', 'scripts/build_gold_v2_dev_set.py': '31c1f17112eea9b279466cdeaa6fdafb43c450be2d6b268136f1b8b8a429f66a', 'scripts/build_q3_missing_occlusion_notebook.py': '290aafd5962f8f5a7230b8733b566258dd09869db72d14fe345733781f32fb2d', 'scripts/build_retrieved_refs.py': '8f27f2aec2d7eecba622f651c2255a4da98ec435bbf9fd6fe2eedca96ddbed93', 'scripts/dev_part_crop_annotation.py': '39de07b82f31778bfb0a94eac89265dd200296f7fa97e2c60ff6308b180682ea', 'scripts/loc_box_annotation.py': '1d839d6a6b10dda3c4c1d3c7ea16aa17e5d6b072f2df90d7e36c4d64fe316cff', 'scripts/q3_multiregion.py': '9eac922fd3ea6cb9164aa8ff8e511ad9a10e665da871e548192c3a53363a2063', 'scripts/run_dev_prompt_diagnostic.py': '76400860bc603002e6c3aa295b212b02bb2f78fd47b0d52f56c34ec2bd8dc8ab', 'scripts/run_gold_baselines.py': '978cb258c7412a767de2b5f7ee9604e726c29dc41922d656084d751d32869575', 'scripts/run_gold_v2_baselines.py': '72585045062bfc5f5ec5243ca50b34ba7fe4bb871f400bd734712354caec9a2a', 'scripts/run_gold_v2_consistency_metrics.py': '283ab1b1c2c1355895a0a047b9da5cfaef9ed67bf55f0108a0cfd6912e4f2dc2', 'scripts/run_gold_v2_dev_baselines.py': '809eca7a79e59f546107409f2a22f2bd1fda2cef3c77b8ea091b0cd401770fa9', 'scripts/run_gold_v2_q23_candidate.py': '19357ad3507dffd33ff5cd97d7febe36c87ab80a255e09398e380770c8d68499', 'scripts/run_gold_v2_q23_zoom_prompt_dev.py': 'cc8b7937fa75250b99af39ea5fa115cae06cb6f5aa470d0c28f417c4ed246d8a', 'scripts/run_gold_v2_q2_q3_agent_dev.py': '0dec0fd461b115de229913463ed7495552bcfb37a548dea0662f212800d695c8', 'scripts/run_gold_v2_q2_q3_refined_dev.py': '577837b328e7b1a9c8b0ea5a06638a798a9f043d84d58f7e085302b531ba8682', 'scripts/run_gold_v2_reference_ablation.py': 'fff6825808e7b9869c42ec2f54438cb98917c442fcdd2e6797d4554daaed9234', 'scripts/run_gold_v2_verify_agent.py': '0963c6c0c0e19c3bffd7fd7b68dcbd54ade6c51ba64bfd04f41fb0fafe5f966a', 'scripts/run_missing_component_probe.py': 'fcb36a2c74d5afaf2f9a1ffdd08a05660b37144cde8ed87d89ba55174b1adfc3', 'scripts/run_q2_conflict_evidence_dev.py': '52ccb22309b3771ef370bdfc271256c8eb39c603f6833900b72e6dd1a9f381c2', 'scripts/run_q3_missing_occlusion_dev.py': '563c85233ab7a272a5323f12732e8649a50678e7ab73903eab71e30224d9ea21', 'scripts/run_selective_missing_expert_agent_dev.py': 'd7dde258c09b6c8d4408226d2d1927616e240185033aabd985d14ab38f6f3288', 'src/insplad_benchmark/__init__.py': '5d9678c34324691fa8d7d5b507501945bb6eafd32e53ef6dedf1979e005cca49', 'src/insplad_benchmark/dataset.py': 'f724c3c5a76f4caa8fa86ac3d3fddef8fed0c466943f979b815f380fbd46652c', 'src/insplad_benchmark/hf_adapters.py': 'dbf178f1a6263c371fc8c7db192a33bdd252910b63267583d5bc24db11f684ab', 'src/insplad_benchmark/metrics.py': 'bb9386501a38031f3d5993744f02a4a1beeb3610bf2a1daa659100511f04cc9d'}
SOURCE_BYTES=base64.b64decode('...')
assert hashlib.sha256(SOURCE_BYTES).hexdigest()==SOURCE_SHA256
CODE=Path('/content/q3_missing_occlusion_'+SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    assert set(archive.namelist())==set(SOURCE_MANIFEST)
    for name in archive.namelist():
        target=CODE/name;content=archive.read(name)
        assert target.resolve().is_relative_to(CODE.resolve())
        assert hashlib.sha256(content).hexdigest()==SOURCE_MANIFEST[name]
        if target.exists() and target.read_bytes()!=content:raise RuntimeError('내장 코드와 다른 런타임 파일; 새 런타임을 사용하세요.')
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
sys.path[:0]=[str(CODE/'scripts'),str(CODE/'src')]
import run_q3_missing_occlusion_dev as experiment
DATASET=CODE/'inputs/dataset.json';DETECTOR=CODE/'inputs/frozen_detector.json'
OUT=DRIVE/RUN_FOLDER
assert OUT.name.startswith(experiment.PREFIX)
print('Q3 후보 부위 실제 실험: 같은23장, 최대414 DINO 특징 +46 LVLM 호출. SOURCE',SOURCE_SHA256)

In [ ]:
import subprocess
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory<20*1024**3:
    raise RuntimeError('L4/A100 VRAM20GB 이상 런타임을 선택하세요.')
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.14.0','accelerate==1.14.0',
    'sentencepiece==0.2.2','numpy>=1.26','pillow>=10','timm'],check=True)
def find_zip(name,size,override):
    if override:
        path=Path(override)
        if not path.is_file() or path.stat().st_size!=size:raise FileNotFoundError(str(path))
        return path
    for path in [DRIVE/name,DRIVE/'lvlm_insplad_data'/name]:
        if path.is_file() and path.stat().st_size==size:return path
    matches=[p for p in DRIVE.rglob(name) if p.is_file() and p.stat().st_size==size]
    if not matches:raise FileNotFoundError(name)
    return sorted(matches)[0]
SUP=find_zip('supervised_fault_classification.zip',874753167,SUP_ZIP_PATH)
UNSUP=find_zip('unsupervised_anomaly_detection.zip',1170967719,UNSUP_ZIP_PATH)
from huggingface_hub import snapshot_download
snapshot=snapshot_download(experiment.prior.MODEL_ID,revision=experiment.prior.MODEL_REVISION)
command=[sys.executable,str(CODE/'scripts/run_q3_missing_occlusion_dev.py'),
    '--dataset',str(DATASET),'--detector',str(DETECTOR),'--sup-zip',str(SUP),'--unsup-zip',str(UNSUP),
    '--model-path',snapshot,'--out-dir',str(OUT)]
LOG=DRIVE/'q3_missing_occlusion_dev240_console.log'
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with LOG.open('a',encoding='utf-8') as log:
    for line in process.stdout:print(line,end='');log.write(line);log.flush()
RETURN_CODE=process.wait();print('RUNNER_RETURN_CODE',RETURN_CODE)

In [ ]:
from google.colab import files
RESULT=DRIVE/'q3_missing_occlusion_dev240_results.zip'
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as archive:
    if OUT.exists():
        for path in sorted(OUT.rglob('*')):
            if path.is_file() and '_images' not in path.relative_to(OUT).parts:
                archive.write(path,'run/'+path.relative_to(OUT).as_posix())
    archive.writestr('source_bundle.zip',SOURCE_BYTES)
    archive.writestr('source_manifest.json',json.dumps(SOURCE_MANIFEST,indent=2))
    archive.writestr('runner_return_code.json',json.dumps({'return_code':RETURN_CODE}))
    if LOG.exists():archive.write(LOG,'console.log')
status=json.loads((OUT/'run_status.json').read_text()) if (OUT/'run_status.json').exists() else {'complete':False}
print('RUN STATUS',status)
if RETURN_CODE or not status.get('complete'):print('실행 오류가 있습니다. ZIP의 원시 기록을 확인해야 합니다.')
elif status.get('invalid_new_outputs'):print('평가 완료이나 무효 출력이 있어 오답으로 포함했습니다.')
print('ZIP',RESULT,'SHA256',hashlib.sha256(RESULT.read_bytes()).hexdigest())
files.download(str(RESULT))